# Clase 5 — Repositorio y gestión de prompts

Un prompt bien diseñado es un activo. Si lo usás una vez y lo perdés, la próxima vez empezás desde cero. En esta clase vamos a construir un sistema simple pero funcional para **organizar, versionar y documentar** prompts de forma que sean reutilizables por vos y por tu equipo.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Por qué documentar prompts |
| 3 | Estructura de metadata de un prompt |
| 4 | Construir el repositorio como DataFrame |
| 5 | Buscar, comparar y versionar prompts |
| 6 | Actividad: construir tu repositorio personal |

---
## 1. Configuración del entorno

**Si es tu primera vez en este curso:**
1. Obtené tu API key en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. Guardala en `.env`:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. Si no querés crear el archivo, la celda te la pide de forma interactiva.

In [1]:
import os
import getpass
import json
import pandas as pd
from datetime import date

BACKEND = "ollama"    # "gemini", "ollama", "local"
GEMINI_MODEL = "gemini-2.5-flash-lite"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini: ")

print(f"Backend: {BACKEND}")

Backend: ollama


In [2]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

elif BACKEND == "ollama":
    import ollama
    OLLAMA_MODEL = "gemma2:9b"
    print("🚀 Conectando a Ollama...")
    try:
        ollama.list()
        print(f"✅ Ollama disponible. Usando modelo: {OLLAMA_MODEL}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Solución: Abre otra terminal y ejecuta: ollama serve")
        raise

elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    ruta_modelo = hf_hub_download(
        repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        filename="qwen2.5-0.5b-instruct-q4_k_m.gguf"
    )
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=200):
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()

    elif BACKEND == "ollama":
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            options={
                "temperature": temperature,
                "num_predict": max_tokens
            }
        )
        return response['message']['content'].strip()

    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

🚀 Conectando a Ollama...
✅ Ollama disponible. Usando modelo: gemma2:9b
Entorno listo.


---
## 2. Por qué documentar prompts

Cuando un equipo empieza a usar modelos de lenguaje en su trabajo, normalmente pasa esto:

1. Cada persona tiene sus propios prompts en notas sueltas o en el historial del chat.
2. Nadie sabe qué versión de un prompt funciona mejor.
3. Cuando alguien se va del equipo, se lleva el conocimiento.
4. Nadie puede mejorar sistemáticamente algo que no está registrado.

Un repositorio de prompts resuelve estos cuatro problemas. No necesita ser sofisticado: alcanza con un formato consistente y el hábito de registrar lo que funciona.

> 💡 Un prompt documentado con contexto vale mucho más que un prompt suelto, aunque el texto sea idéntico. El contexto explica *por qué* funciona.

---
## 3. Estructura de metadata de un prompt

Cada prompt en el repositorio tiene asociada una ficha con estos campos:

| Campo | Tipo | Descripción |
|---|---|---|
| `id` | string | Identificador único (ej: `feedback-v1`) |
| `tarea` | string | Categoría de la tarea (ej: `redacción`, `análisis`, `clasificación`) |
| `descripcion` | string | Qué hace el prompt en una oración |
| `version` | int | Número de versión, empieza en 1 |
| `modelo` | string | Modelo con el que fue probado |
| `temperatura` | float | Temperatura usada en las pruebas |
| `score` | int (1–5) | Calidad subjetiva de los resultados |
| `fecha` | string | Fecha de última actualización |
| `notas` | string | Qué se mejoró, qué no funciona, advertencias |

In [3]:
# ─── Estructura de un prompt documentado ─────────────────────────────────────
# Cada entrada es un diccionario con el texto del prompt y su metadata.

def crear_entrada(id, tarea, descripcion, prompt_texto, system_prompt,
                  temperatura, score, notas, version=1, modelo=None):
    """Crea una entrada estandarizada para el repositorio."""
    return {
        "id":           id,
        "tarea":        tarea,
        "descripcion":  descripcion,
        "prompt":       prompt_texto,
        "system":       system_prompt,
        "version":      version,
        "modelo":       modelo or GEMINI_MODEL if BACKEND == "gemini" else "local",
        "temperatura":  temperatura,
        "score":        score,
        "fecha":        str(date.today()),
        "notas":        notas,
    }


print("Función crear_entrada lista.")

Función crear_entrada lista.


---
## 4. Construir el repositorio como DataFrame

Vamos a cargar el repositorio con varios prompts representativos de distintas categorías. Algunos son versiones anteriores (`v1`) de prompts que después mejoramos (`v2`), para demostrar el versionado.

In [4]:
# ─── Prompts de ejemplo para el repositorio ───────────────────────────────────

repositorio = [
    crear_entrada(
        id="feedback-v1",
        tarea="redacción",
        descripcion="Feedback constructivo para empleado — versión inicial",
        prompt_texto="Escribí un feedback para {nombre} sobre {situacion}.",
        system_prompt="Sos un líder de equipo.",
        temperatura=0.7, score=2,
        notas="Demasiado genérico. No especifica estructura ni tono."
    ),
    crear_entrada(
        id="feedback-v2",
        tarea="redacción",
        descripcion="Feedback constructivo para empleado — con CoT y rúbrica",
        prompt_texto="""Escribí feedback para {nombre}. Situación: {situacion}
Criterios: primero lo positivo, luego el área de mejora como hecho (no juicio),
proponer una acción concreta. Tono directo. Máximo 5 oraciones.""",
        system_prompt="Sos un líder de equipo con experiencia en desarrollo de personas.",
        temperatura=0.5, score=5,
        notas="Versión estable. Probado en 12 casos reales con buen resultado.",
        version=2
    ),
    crear_entrada(
        id="sentiment-v1",
        tarea="clasificación",
        descripcion="Clasificación de sentimiento de comentarios de clientes",
        prompt_texto="""Clasificá el sentimiento: '{comentario}'
Formato: Sentimiento: <Positivo|Negativo|Neutro> | Razón: <una frase>""",
        system_prompt="Sos un analista de experiencia del cliente.",
        temperatura=0.3, score=4,
        notas="Funciona bien. Occasionally clasifica como Neutro casos borderline Negativo."
    ),
    crear_entrada(
        id="resumen-ejecutivo-v1",
        tarea="resumen",
        descripcion="Resumen ejecutivo de reportes o documentos extensos",
        prompt_texto="""Resumí el siguiente texto en formato ejecutivo:
1. Objetivo principal (1 oración)
2. Hallazgos clave (3 bullets)
3. Próximos pasos recomendados (2 bullets)

Texto: {texto}""",
        system_prompt="Sos un consultor de management que comunica con claridad.",
        temperatura=0.4, score=4,
        notas="Muy útil para reuniones. Ajustar número de bullets según longitud del texto."
    ),
    crear_entrada(
        id="sql-v1",
        tarea="código",
        descripcion="Generar consultas SQL a partir de preguntas en lenguaje natural",
        prompt_texto="""Generá una consulta SQL para esta necesidad: '{necesidad}'
Esquema disponible: {esquema}
Reglas: solo SELECT, sin subconsultas anidadas, incluir comentario de la lógica.""",
        system_prompt="Sos un DBA que escribe SQL claro y eficiente.",
        temperatura=0.2, score=4,
        notas="Temperatura baja para mayor determinismo. Validar siempre antes de ejecutar."
    ),
]

# Convertir a DataFrame para visualización y búsqueda cómodas
df = pd.DataFrame(repositorio)
print(f"Repositorio cargado: {len(df)} prompts")
df[["id", "tarea", "descripcion", "version", "score", "fecha"]]

Repositorio cargado: 5 prompts


,id,tarea,descripcion,version,score,fecha
0,feedback-v1,redacción,Feedback constructivo para empleado — versión ...,1,2,2026-05-20
1,feedback-v2,redacción,Feedback constructivo para empleado — con CoT ...,2,5,2026-05-20
2,sentiment-v1,clasificación,Clasificación de sentimiento de comentarios de...,1,4,2026-05-20
3,resumen-ejecutivo-v1,resumen,Resumen ejecutivo de reportes o documentos ext...,1,4,2026-05-20
4,sql-v1,código,Generar consultas SQL a partir de preguntas en...,1,4,2026-05-20


---
## 5. Buscar, comparar y versionar prompts

In [5]:
# ─── Buscar por tarea ─────────────────────────────────────────────────────────
def buscar_por_tarea(df, tarea):
    resultado = df[df["tarea"] == tarea][["id", "descripcion", "version", "score", "notas"]]
    print(f"Prompts para tarea '{tarea}':")
    print(resultado.to_string(index=False))


buscar_por_tarea(df, "redacción")

Prompts para tarea 'redacción':
         id                                             descripcion  version  score                                                           notas
feedback-v1   Feedback constructivo para empleado — versión inicial        1      2           Demasiado genérico. No especifica estructura ni tono.
feedback-v2 Feedback constructivo para empleado — con CoT y rúbrica        2      5 Versión estable. Probado en 12 casos reales con buen resultado.


In [6]:
# ─── Recuperar el mejor prompt de una tarea ───────────────────────────────────
def obtener_mejor_prompt(df, tarea):
    """Devuelve el prompt con mayor score para una tarea dada."""
    subset = df[df["tarea"] == tarea]
    if subset.empty:
        return None
    mejor = subset.loc[subset["score"].idxmax()]
    return mejor


mejor_feedback = obtener_mejor_prompt(df, "redacción")
print("Mejor prompt de redacción:")
print(f"  ID:      {mejor_feedback['id']}")
print(f"  Score:   {mejor_feedback['score']}")
print(f"  Notas:   {mejor_feedback['notas']}")
print(f"  Prompt:  {mejor_feedback['prompt'][:100]}...")

Mejor prompt de redacción:
  ID:      feedback-v2
  Score:   5
  Notas:   Versión estable. Probado en 12 casos reales con buen resultado.
  Prompt:  Escribí feedback para {nombre}. Situación: {situacion}
Criterios: primero lo positivo, luego el área...


In [7]:
# ─── Ejecutar un prompt del repositorio con variables ─────────────────────────
def ejecutar_desde_repo(df, prompt_id, variables: dict, max_tokens=200):
    """Busca el prompt por ID, rellena las variables y lo ejecuta."""
    fila = df[df["id"] == prompt_id]
    if fila.empty:
        raise ValueError(f"No existe un prompt con ID '{prompt_id}'")

    entrada = fila.iloc[0]
    texto   = entrada["prompt"].format(**variables)   # rellena las {variables}
    sistema = entrada["system"]
    temp    = entrada["temperatura"]

    print(f"Ejecutando: {prompt_id} (v{entrada['version']}, score {entrada['score']})")
    print("-" * 50)
    return llamar_llm(texto, system_prompt=sistema, temperature=temp, max_tokens=max_tokens)


resultado = ejecutar_desde_repo(
    df,
    prompt_id="feedback-v2",
    variables={
        "nombre": "Martín",
        "situacion": "llegó tarde a tres reuniones esta semana pero su trabajo fue impecable"
    }
)
print(resultado)

Ejecutando: feedback-v2 (v2, score 5)
--------------------------------------------------
Martín, tu trabajo esta semana ha sido excelente, la calidad es impecable y siempre entregas proyectos a tiempo. No obstante, te has llegado tarde a tres reuniones este semana. Para asegurarnos de que todos estamos en la misma página, propongo que nos pongamos en contacto antes de las reuniones si se presenta algún imprevisto.


In [8]:
# ─── Exportar el repositorio a JSON ───────────────────────────────────────────
# Guardamos el repositorio como archivo para compartir o versionar con git.

RUTA_REPO = "prompt_repositorio.json"

with open(RUTA_REPO, "w", encoding="utf-8") as f:
    json.dump(repositorio, f, ensure_ascii=False, indent=2)

print(f"Repositorio exportado a: {RUTA_REPO}")
print(f"Tamaño: {os.path.getsize(RUTA_REPO)} bytes")

Repositorio exportado a: prompt_repositorio.json
Tamaño: 2882 bytes


In [9]:
# ─── Importar y agregar una nueva versión de un prompt ────────────────────────
# Simulamos una mejora del prompt de clasificación de sentimiento.

with open(RUTA_REPO, encoding="utf-8") as f:
    repo_cargado = json.load(f)

# Agregar nueva versión mejorada
repo_cargado.append(
    crear_entrada(
        id="sentiment-v2",
        tarea="clasificación",
        descripcion="Clasificación de sentimiento — con confianza y acción recomendada",
        prompt_texto="""Clasificá el sentimiento de este comentario de cliente: '{comentario}'
Formato estricto:
Sentimiento: <Positivo|Negativo|Neutro>
Confianza: <Alta|Media|Baja>
Acción sugerida: <una oración sobre qué debería hacer el equipo de soporte>""",
        system_prompt="Sos un analista de experiencia del cliente con foco en acciones preventivas.",
        temperatura=0.3, score=5,
        notas="Agrega confianza y acción recomendada. Más útil para triaje de tickets.",
        version=2
    )
)

# Guardar versión actualizada
with open(RUTA_REPO, "w", encoding="utf-8") as f:
    json.dump(repo_cargado, f, ensure_ascii=False, indent=2)

df_actualizado = pd.DataFrame(repo_cargado)
print(f"Repositorio actualizado: {len(df_actualizado)} prompts")
df_actualizado[["id", "tarea", "version", "score"]]

Repositorio actualizado: 6 prompts


,id,tarea,version,score
0,feedback-v1,redacción,1,2
1,feedback-v2,redacción,2,5
2,sentiment-v1,clasificación,1,4
3,resumen-ejecutivo-v1,resumen,1,4
4,sql-v1,código,1,4
5,sentiment-v2,clasificación,2,5


---
## 6. Actividad: construir tu repositorio personal

Tomá los prompts que diseñaste en las clases anteriores (anatomía, errores y patrones, few-shot, CoT) y registralos en tu repositorio. Mínimo: 5 prompts de al menos 3 categorías distintas.

In [36]:
# TODO: Completá tu repositorio con los prompts de las clases anteriores
# Usá la función crear_entrada() para cada uno.
# ============================================================================
# MI REPOSITORIO DE PROMPTS - SISTEMA DE ATENCIÓN AL CLIENTE
# ============================================================================

mi_repositorio = [
    
    # ========================================================================
    # ETAPA 1: CLASIFICACIÓN DE MENSAJES (Few-Shot)
    # ========================================================================
    
    crear_entrada(
        id="clasificacion-tipo-v1",
        tarea="clasificacion",
        descripcion="Clasificar mensaje por tipo de problema usando Few-Shot",
        prompt_texto="""Clasificá el siguiente mensaje de cliente según su tipo.

EJEMPLOS:
---
Mensaje: "Hace 3 días que no tengo internet y trabajo desde casa"
Tipo: Técnico

Mensaje: "Me cobraron dos veces este mes en la tarjeta"
Tipo: Cobranzas

Mensaje: "Necesito que vengan a instalar el servicio en mi nueva dirección"
Tipo: Instalación

Mensaje: "La factura tiene un cargo que no reconozco por $5000"
Tipo: Facturación

Mensaje: "Quiero dar de baja el servicio a partir del próximo mes"
Tipo: Bajas

Mensaje: "Necesito cambiar la titularidad del servicio a mi nombre"
Tipo: Administrativo
---

MENSAJE A CLASIFICAR:
"{mensaje}"

RESPONDE SOLO CON UNA PALABRA: Técnico, Cobranzas, Instalación, Facturación, Bajas o Administrativo""",
        system_prompt="Sos un clasificador experto de mensajes de clientes de telecomunicaciones. Analizás el contenido y asignás la categoría correcta.",
        temperatura=0.1,
        score=5,
        notas="Temperatura muy baja para respuestas consistentes. Few-Shot con 6 ejemplos claros."
    ),
    
    crear_entrada(
        id="clasificacion-naturaleza-v1",
        tarea="clasificacion",
        descripcion="Clasificar mensaje por naturaleza (Reclamo/Solicitud/Consulta)",
        prompt_texto="""Clasificá la naturaleza del siguiente mensaje de cliente.

EJEMPLOS:
---
Mensaje: "Hace 3 días que no tengo internet y necesito que lo arreglen YA"
Naturaleza: Reclamo

Mensaje: "¿Podrían enviarme la factura del mes pasado?"
Naturaleza: Solicitud

Mensaje: "¿Qué planes de 300 megas tienen disponibles?"
Naturaleza: Consulta

Mensaje: "Me están cobrando de más desde hace 2 meses"
Naturaleza: Reclamo

Mensaje: "Quiero cambiar mi plan actual por uno más económico"
Naturaleza: Solicitud

Mensaje: "¿Hasta cuándo tengo tiempo para pagar sin recargo?"
Naturaleza: Consulta
---

MENSAJE A CLASIFICAR:
"{mensaje}"

RESPONDE SOLO CON UNA PALABRA: Reclamo, Solicitud o Consulta""",
        system_prompt="Sos un analista de comunicaciones que identifica si el cliente está reclamando, solicitando algo o solo consultando.",
        temperatura=0.1,
        score=5,
        notas="Few-Shot con ejemplos claros de cada categoría. Temperatura baja para consistencia."
    ),
    
    crear_entrada(
        id="deteccion-complejidad-v1",
        tarea="clasificacion",
        descripcion="Detectar si un mensaje es complejo y requiere atención humana",
        prompt_texto="""Analizá si este mensaje requiere atención humana por su complejidad.

CRITERIOS PARA MARCAR COMO COMPLEJO:
- Involucra múltiples problemas simultáneos
- Contiene lenguaje agresivo o amenazas legales
- Requiere decisiones que exceden políticas estándar
- Involucra montos altos de dinero (>$50000)
- Menciona problemas de salud, seguridad o emergencias
- Cliente indica frustración extrema o problema reiterado (3+ veces)

EJEMPLOS:
---
Mensaje: "Hace 3 días que no tengo internet"
Complejidad: Simple
Razón: Problema único y estándar

Mensaje: "Esta es la CUARTA VEZ que reclamo. No tengo internet, me cobraron de más y NADIE me soluciona nada. Voy a iniciar acciones legales"
Complejidad: Complejo
Razón: Problema reiterado, múltiples issues, amenaza legal

Mensaje: "Necesito internet YA porque mi madre tiene 80 años y usa telemedicina"
Complejidad: Complejo
Razón: Involucra salud y emergencia

Mensaje: "¿Cuánto sale el plan de 100 megas?"
Complejidad: Simple
Razón: Consulta estándar
---

MENSAJE A ANALIZAR:
"{mensaje}"

FORMATO DE RESPUESTA:
Complejidad: <Simple|Complejo>
Razón: <una oración>""",
        system_prompt="Sos un supervisor de soporte con criterio para identificar casos que necesitan escalamiento a humanos.",
        temperatura=0.2,
        score=5,
        notas="Criterios explícitos de complejidad. Incluye razón para auditar decisiones."
    ),
    
    # ========================================================================
    # ETAPA 2: RESPUESTAS AUTOMÁTICAS (según clasificación)
    # ========================================================================
    
    crear_entrada(
        id="respuesta-tecnico-v1",
        tarea="respuesta",
        descripcion="Respuesta a problemas técnicos con pasos de diagnóstico",
        prompt_texto="""Generá una respuesta para este problema técnico.

TIPO: {tipo}
NATURALEZA: {naturaleza}
MENSAJE: "{mensaje}"

ESTRUCTURA DE RESPUESTA:
1. Empatía (reconocer el problema)
2. Verificación rápida (2-3 pasos que puede hacer el cliente)
3. Próximos pasos (qué haremos nosotros)
4. Tiempo estimado de resolución
5. Cierre con contacto directo

TONO: Profesional, empático, orientado a solución.
MÁXIMO: 6 oraciones.""",
        system_prompt="Sos técnico de soporte nivel 1 con experiencia en diagnóstico remoto de conectividad.",
        temperatura=0.6,
        score=4,
        notas="Balancea empatía con soluciones técnicas. Incluye pasos verificables."
    ),
    
    crear_entrada(
        id="respuesta-cobranzas-v1",
        tarea="respuesta",
        descripcion="Respuesta a reclamos de cobros duplicados o incorrectos",
        prompt_texto="""Generá una respuesta para este reclamo de cobranza.

TIPO: {tipo}
NATURALEZA: {naturaleza}
MENSAJE: "{mensaje}"

ESTRUCTURA DE RESPUESTA:
1. Disculpa y empatía
2. Verificación del reclamo (indicar que se revisará)
3. Acción concreta (reintegro, ajuste, etc.)
4. Plazo específico
5. Compensación si aplica
6. Contacto directo

TONO: Muy empático, responsable, con sentido de urgencia.
MÁXIMO: 6 oraciones.
IMPORTANTE: Si el monto parece alto (>$10000) mencioná que un supervisor revisará personalmente.""",
        system_prompt="Sos especialista en cobranzas y resolución de disputas de facturación, con autoridad para ofrecer compensaciones menores.",
        temperatura=0.5,
        score=4,
        notas="Prioriza empatía y compensación. Incluye escalamiento para montos altos."
    ),
    
    crear_entrada(
        id="respuesta-instalacion-v1",
        tarea="respuesta",
        descripcion="Respuesta a solicitudes de instalación o cambio de domicilio",
        prompt_texto="""Generá una respuesta para esta solicitud de instalación.

TIPO: {tipo}
NATURALEZA: {naturaleza}
MENSAJE: "{mensaje}"

ESTRUCTURA DE RESPUESTA:
1. Confirmación de la solicitud
2. Información que necesitamos (dirección completa, horario disponible)
3. Próximos pasos (visita técnica, verificación de cobertura)
4. Tiempos estimados
5. Cierre con contacto

TONO: Colaborativo, claro, orientado a acción.
MÁXIMO: 5 oraciones.""",
        system_prompt="Sos coordinador de instalaciones con experiencia en logística de campo.",
        temperatura=0.5,
        score=4,
        notas="Estructura clara de pasos. Solicita información necesaria de forma natural."
    ),
    
    crear_entrada(
        id="respuesta-facturacion-v1",
        tarea="respuesta",
        descripcion="Respuesta a consultas sobre cargos en factura",
        prompt_texto="""Generá una respuesta para esta consulta de facturación.

TIPO: {tipo}
NATURALEZA: {naturaleza}
MENSAJE: "{mensaje}"

ESTRUCTURA DE RESPUESTA:
1. Reconocimiento de la consulta
2. Explicación clara del cargo (si es identificable)
3. Ofrecimiento de factura detallada
4. Invitación a revisar juntos si hay duda
5. Contacto directo

TONO: Transparente, educativo, paciente.
MÁXIMO: 5 oraciones.""",
        system_prompt="Sos analista de facturación con habilidad para explicar cargos complejos de forma simple.",
        temperatura=0.5,
        score=4,
        notas="Tono educativo sin ser condescendiente. Proactivo en ofrecer detalles."
    ),
    
    crear_entrada(
        id="respuesta-bajas-v1",
        tarea="respuesta",
        descripcion="Respuesta a solicitudes de baja de servicio",
        prompt_texto="""Generá una respuesta para esta solicitud de baja.

TIPO: {tipo}
NATURALEZA: {naturaleza}
MENSAJE: "{mensaje}"

ESTRUCTURA DE RESPUESTA:
1. Lamentar la decisión (sin ser insistente)
2. Confirmar proceso de baja
3. Informar sobre devolución de equipos si aplica
4. Mencionar liquidación final
5. Ofrecer alternativa solo si tiene sentido (cambio de plan, pausa temporal)
6. Dejar puerta abierta para el futuro

TONO: Respetuoso, no insistente, profesional.
MÁXIMO: 6 oraciones.
IMPORTANTE: No presionar al cliente.""",
        system_prompt="Sos especialista en retención con enfoque en mantener buena relación aunque el cliente se vaya.",
        temperatura=0.6,
        score=4,
        notas="Balance entre retención y respeto. No ser agresivo con ofertas."
    ),
    
    crear_entrada(
        id="respuesta-administrativo-v1",
        tarea="respuesta",
        descripcion="Respuesta a trámites administrativos (cambio titular, etc)",
        prompt_texto="""Generá una respuesta para este trámite administrativo.

TIPO: {tipo}
NATURALEZA: {naturaleza}
MENSAJE: "{mensaje}"

ESTRUCTURA DE RESPUESTA:
1. Confirmación de la solicitud
2. Documentación requerida (lista específica)
3. Formas de envío (mail, presencial, app)
4. Tiempo de procesamiento
5. Contacto para dudas

TONO: Claro, organizado, servicial.
MÁXIMO: 5 oraciones.""",
        system_prompt="Sos asistente administrativo con conocimiento de todos los trámites y documentación requerida.",
        temperatura=0.4,
        score=4,
        notas="Muy específico con requisitos. Evita ambigüedades."
    ),
    
    crear_entrada(
        id="respuesta-complejo-v1",
        tarea="respuesta",
        descripcion="Respuesta de derivación para casos complejos",
        prompt_texto="""Generá una respuesta de derivación a supervisor.

MENSAJE: "{mensaje}"
RAZÓN DE COMPLEJIDAD: {razon_complejidad}

ESTRUCTURA:
1. Reconocimiento del problema con empatía extra
2. Indicar que se asignará un supervisor
3. Tiempo máximo de contacto (2 horas hábiles)
4. Número de caso para tracking
5. Disculpa por la situación

TONO: Muy empático, toma responsabilidad, urgencia.
MÁXIMO: 5 oraciones.""",
        system_prompt="Sos el sistema de escalamiento que asegura que casos complejos reciban atención prioritaria.",
        temperatura=0.5,
        score=5,
        notas="Enfoque en empatía y seguimiento. Genera número de caso único."
    ),

  crear_entrada(
    id="clasificacion-completa-v4",
    tarea="clasificacion-completa",
    descripcion="Clasificación con 4 tipos (Cobranzas y Facturación unificados en Administrativo)",
    prompt_texto="""Analizá este mensaje de cliente y clasificalo en las tres dimensiones.

EJEMPLOS DE CLASIFICACIÓN COMPLETA:

=== TÉCNICO ===
Mensaje: "Hace 3 días que no tengo internet y trabajo desde casa"
Clasificacion: Reclamo - Técnico - Simple
Razón: Problema único estándar sin agravantes

Mensaje: "¿Cómo configuro el router nuevo que me enviaron?"
Clasificacion: Consulta - Técnico - Simple
Razón: Consulta técnica estándar sin urgencia

Mensaje: "Podrían enviar un técnico para revisar la señal WiFi"
Clasificacion: Solicitud - Técnico - Simple
Razón: Solicitud de visita técnica preventiva

Mensaje: "Es la QUINTA VEZ que se corta internet. Tengo reuniones importantes y NADIE soluciona nada"
Clasificacion: Reclamo - Técnico - Complejo
Razón: Problema reiterado (5ta vez), impacto laboral, frustración extrema

=== ADMINISTRATIVO (incluye facturación, cobranzas, pagos, planes, descuentos, cambios de titular, trámites) ===
Mensaje: "Me cobraron dos veces este mes en la tarjeta"
Clasificacion: Reclamo - Administrativo - Simple
Razón: Error de cobro común, monto no especificado

Mensaje: "¿Puedo pagar en cuotas la deuda acumulada?"
Clasificacion: Consulta - Administrativo - Simple
Razón: Consulta sobre opciones de pago

Mensaje: "Necesito que me envíen un plan de pagos por los $8000 que debo"
Clasificacion: Solicitud - Administrativo - Simple
Razón: Solicitud proactiva de arreglo de pago

Mensaje: "La factura tiene un cargo de 'Servicio Premium' que no reconozco"
Clasificacion: Reclamo - Administrativo - Simple
Razón: Reclamo por cargo no reconocido en factura

Mensaje: "¿Qué incluye el cargo de 'Mantenimiento' en la factura?"
Clasificacion: Consulta - Administrativo - Simple
Razón: Consulta sobre conceptos de facturación

Mensaje: "¿Pueden enviarme las facturas de los últimos 6 meses?"
Clasificacion: Solicitud - Administrativo - Simple
Razón: Solicitud de documentación

Mensaje: "Necesito cambiar la titularidad del servicio a mi nombre"
Clasificacion: Solicitud - Administrativo - Simple
Razón: Trámite administrativo estándar

Mensaje: "¿Qué documentos necesito para cambiar el domicilio de facturación?"
Clasificacion: Consulta - Administrativo - Simple
Razón: Consulta sobre requisitos de trámite

Mensaje: "Necesito un descuento porque soy jubilado"
Clasificacion: Solicitud - Administrativo - Simple
Razón: Solicitud de beneficio especial

Mensaje: "Me cobraron $45000 de más hace 3 meses, reclamo, nadie responde y ahora amenazan con corte. Voy a COPREC"
Clasificacion: Reclamo - Administrativo - Complejo
Razón: Monto muy alto, problema antiguo sin resolver, amenaza legal, riesgo de corte

Mensaje: "Hace 6 meses que reclamo por facturas con cargos incorrectos que suman $60000 y NADIE me responde"
Clasificacion: Reclamo - Administrativo - Complejo
Razón: Reclamo de facturación reiterado, monto alto, sin respuesta prolongada

Mensaje: "Envié los documentos para el cambio de titular hace 2 semanas y no lo procesaron"
Clasificacion: Reclamo - Administrativo - Simple
Razón: Demora en trámite sin urgencia extrema

=== INSTALACIÓN ===
Mensaje: "Me mudé y necesito que instalen el servicio en mi nueva dirección"
Clasificacion: Solicitud - Instalación - Simple
Razón: Solicitud estándar de mudanza

Mensaje: "¿Cuánto demora la instalación en un barrio nuevo?"
Clasificacion: Consulta - Instalación - Simple
Razón: Consulta sobre tiempos de instalación

Mensaje: "El técnico de instalación no vino en el horario acordado"
Clasificacion: Reclamo - Instalación - Simple
Razón: Incumplimiento de horario sin agravantes

Mensaje: "Hace 15 días pedí instalación, vinieron 2 veces y no terminaron. Trabajo remoto y no puedo seguir esperando"
Clasificacion: Reclamo - Instalación - Complejo
Razón: Instalación fallida reiterada, impacto laboral, demora excesiva

=== BAJAS ===
Mensaje: "Quiero dar de baja el servicio porque me mudo al exterior"
Clasificacion: Solicitud - Bajas - Simple
Razón: Solicitud de baja con motivo válido

Mensaje: "¿Hay penalidad si doy de baja antes del año?"
Clasificacion: Consulta - Bajas - Simple
Razón: Consulta sobre condiciones de baja

Mensaje: "Pedí la baja hace una semana y todavía no recibí la confirmación"
Clasificacion: Reclamo - Bajas - Simple
Razón: Falta de confirmación sin urgencia extrema

Mensaje: "Pedí la baja hace un mes y me siguen cobrando. Si no lo solucionan HOY inicio acciones legales"
Clasificacion: Reclamo - Bajas - Complejo
Razón: Baja no procesada, cobros indebidos posteriores, amenaza legal, urgencia extrema

---

MENSAJE A CLASIFICAR:
"{mensaje}"

FORMATO DE RESPUESTA (respetá exactamente este formato en DOS líneas):
Clasificacion: <Naturaleza> - <Tipo> - <Complejidad>
Razón: <una oración explicando la complejidad>

Donde:
- Naturaleza: Reclamo, Solicitud o Consulta
- Tipo: Técnico, Administrativo, Instalación o Bajas
- Complejidad: Simple o Complejo""",
    system_prompt="Sos un clasificador experto. Administrativo incluye TODO lo relacionado con facturación, cobranzas, pagos, planes de pago, descuentos, extensiones, cambios de titular y trámites. Respondés exactamente en el formato solicitado.",
    temperatura=0.1,
    score=5,
    notas="v4 final: 4 tipos (Cobranzas y Facturación unificados en Administrativo). Formato: Naturaleza - Tipo - Complejidad."
),
crear_entrada(
    id="respuesta-automatica-v1",
    tarea="respuesta-enfocada",
    descripcion="Genera respuesta automática basada en la clasificación del mensaje",
    prompt_texto="""Generá una respuesta para el cliente basándote en la clasificación del mensaje.

MENSAJE ORIGINAL: "{mensaje}"
CLASIFICACIÓN: {clasificacion}
RAZÓN: {razon}

INSTRUCCIONES SEGÚN TIPO:

=== SI ES TÉCNICO ===
1. Empatía por el problema técnico
2. Pasos rápidos de verificación (2-3 acciones simples que puede hacer)
3. Qué haremos nosotros (revisión, envío de técnico)
4. Tiempo estimado de resolución
5. Contacto directo si persiste

=== SI ES ADMINISTRATIVO (Facturación/Cobranzas/Trámites) ===
1. Reconocimiento del tema (factura/pago/trámite)
2. Verificación que haremos (revisar cuenta, procesar documentos)
3. Acción concreta (ajuste, reintegro, procesamiento)
4. Plazo específico (24-48hs para consultas, 2-5 días para trámites)
5. Contacto para seguimiento

=== SI ES INSTALACIÓN ===
1. Confirmación de la solicitud
2. Información que necesitamos (dirección, disponibilidad horaria)
3. Próximos pasos (verificar cobertura, agendar visita)
4. Tiempos estimados (2-5 días hábiles)
5. Contacto para coordinar

=== SI ES BAJAS ===
1. Lamentar la decisión (sin ser insistente)
2. Confirmar el proceso de baja
3. Información sobre equipos (devolución si aplica)
4. Liquidación final
5. Dejar puerta abierta profesionalmente

AJUSTES SEGÚN COMPLEJIDAD:

SI ES SIMPLE:
- Tono: profesional, directo, orientado a solución
- Extensión: 4-5 oraciones
- Enfoque: resolver rápido

SI ES COMPLEJO:
- Tono: muy empático, responsable, urgente
- Extensión: 5-6 oraciones
- Enfoque: priorizar, escalar, dar seguimiento personal
- Mencionar que un supervisor revisará personalmente
- Ofrecer contacto directo prioritario

FORMATO DE RESPUESTA:
- Máximo 6 oraciones
- Tono profesional y empático
- Sin jerga técnica innecesaria
- Incluir próximos pasos claros
- Terminar con forma de contacto

RESPUESTA:""",
    system_prompt="Sos el sistema de respuestas automáticas de atención al cliente. Generás respuestas profesionales, empáticas y orientadas a solución según la clasificación del mensaje.",
    temperatura=0.6,
    score=5,
    notas="Genera respuestas automáticas basadas en clasificación v4. Ajusta tono y contenido según tipo y complejidad."
),
]

# ============================================================================
# MOSTRAR REPOSITORIO
# ============================================================================

df_mio = pd.DataFrame(mi_repositorio) if mi_repositorio else pd.DataFrame()
print(f"Mi repositorio: {len(mi_repositorio)} prompts")
if not df_mio.empty:
    print(df_mio[["id", "tarea", "descripcion", "score"]].to_string(index=False))

Mi repositorio: 12 prompts
                         id                  tarea                                                                      descripcion  score
      clasificacion-tipo-v1          clasificacion                          Clasificar mensaje por tipo de problema usando Few-Shot      5
clasificacion-naturaleza-v1          clasificacion                   Clasificar mensaje por naturaleza (Reclamo/Solicitud/Consulta)      5
   deteccion-complejidad-v1          clasificacion                    Detectar si un mensaje es complejo y requiere atención humana      5
       respuesta-tecnico-v1              respuesta                          Respuesta a problemas técnicos con pasos de diagnóstico      4
     respuesta-cobranzas-v1              respuesta                          Respuesta a reclamos de cobros duplicados o incorrectos      4
   respuesta-instalacion-v1              respuesta                     Respuesta a solicitudes de instalación o cambio de domicilio      4


In [ ]:
# TODO: Ejecutá al menos un prompt de tu repositorio con datos reales
# Usá ejecutar_desde_repo() o llamar_llm() directamente.

# Ejemplo:
# print(ejecutar_desde_repo(df_mio, "tu-prompt-id", {"variable": "valor"}))

---
## Entregable

Guardá el notebook con las celdas ejecutadas.
El entregable es el `mi_repositorio` completo (mínimo 5 prompts, 3 categorías) más al menos una ejecución real con datos.

**Este repositorio es la base del Proyecto de Módulo:** sistema de predicción simple + guía de prompts optimizados.

**Para la próxima clase:** arrancamos con Machine Learning — cuándo usarlo y cómo entrenar tu primer modelo.

In [27]:
# ============================================================================
# PRUEBAS COMPLETAS V4: 4 TIPOS FINALES
# ============================================================================

print("\n" + "="*80)
print("PRUEBAS EXHAUSTIVAS V4 - 4 TIPOS (Cobranzas y Facturación → Administrativo)")
print("="*80)

# ============================================================================
# TÉCNICO (4 casos)
# ============================================================================
print("\n" + "="*80)
print(">>> TIPO: TÉCNICO")
print("="*80)

print("\n--- Reclamo | Técnico | Simple ---")
print("Mensaje: Hace 2 días que no tengo internet")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Hace 2 días que no tengo internet'}, max_tokens=150))
print("-" * 80)

print("\n--- Reclamo | Técnico | Complejo ---")
print("Mensaje: Es la QUINTA VEZ que se corta internet. Tengo reuniones importantes y NADIE soluciona nada")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Es la QUINTA VEZ que se corta internet. Tengo reuniones importantes y NADIE soluciona nada'}, max_tokens=150))
print("-" * 80)

print("\n--- Solicitud | Técnico | Simple ---")
print("Mensaje: Podrían enviar un técnico para revisar la señal WiFi")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Podrían enviar un técnico para revisar la señal WiFi'}, max_tokens=150))
print("-" * 80)

print("\n--- Consulta | Técnico | Simple ---")
print("Mensaje: ¿Cómo configuro el router nuevo que me enviaron?")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': '¿Cómo configuro el router nuevo que me enviaron?'}, max_tokens=150))
print("-" * 80)

# ============================================================================
# ADMINISTRATIVO (incluye COBRANZAS + FACTURACIÓN - 12 casos)
# ============================================================================
print("\n" + "="*80)
print(">>> TIPO: ADMINISTRATIVO (incluye Cobranzas + Facturación)")
print("="*80)

# COBRANZAS
print("\n--- Reclamo | Administrativo | Simple (Cobro duplicado) ---")
print("Mensaje: Me cobraron dos veces este mes en la tarjeta")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Me cobraron dos veces este mes en la tarjeta'}, max_tokens=150))
print("-" * 80)

print("\n--- Reclamo | Administrativo | Complejo (Cobro + amenaza legal) ---")
print("Mensaje: Me cobraron $45000 de más hace 3 meses, reclamo, nadie responde y ahora amenazan con corte. Voy a COPREC")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Me cobraron $45000 de más hace 3 meses, reclamo, nadie responde y ahora amenazan con corte. Voy a COPREC'}, max_tokens=150))
print("-" * 80)

print("\n--- Solicitud | Administrativo | Simple (Plan de pagos) ---")
print("Mensaje: Necesito que me envíen un plan de pagos por los $8000 que debo")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Necesito que me envíen un plan de pagos por los $8000 que debo'}, max_tokens=150))
print("-" * 80)

print("\n--- Consulta | Administrativo | Simple (Opciones de pago) ---")
print("Mensaje: ¿Puedo pagar en cuotas la deuda acumulada?")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': '¿Puedo pagar en cuotas la deuda acumulada?'}, max_tokens=150))
print("-" * 80)

# FACTURACIÓN
print("\n--- Reclamo | Administrativo | Simple (Cargo no reconocido) ---")
print("Mensaje: La factura tiene un cargo de 'Servicio Premium' que no reconozco")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': "La factura tiene un cargo de 'Servicio Premium' que no reconozco"}, max_tokens=150))
print("-" * 80)

print("\n--- Reclamo | Administrativo | Complejo (Facturación reiterada) ---")
print("Mensaje: Hace 6 meses que reclamo por facturas con cargos incorrectos que suman $60000 y NADIE me responde")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Hace 6 meses que reclamo por facturas con cargos incorrectos que suman $60000 y NADIE me responde'}, max_tokens=150))
print("-" * 80)

print("\n--- Solicitud | Administrativo | Simple (Envío de facturas) ---")
print("Mensaje: ¿Pueden enviarme las facturas de los últimos 6 meses?")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': '¿Pueden enviarme las facturas de los últimos 6 meses?'}, max_tokens=150))
print("-" * 80)

print("\n--- Consulta | Administrativo | Simple (Concepto factura) ---")
print("Mensaje: ¿Qué incluye el cargo de 'Mantenimiento' en la factura?")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': "¿Qué incluye el cargo de 'Mantenimiento' en la factura?"}, max_tokens=150))
print("-" * 80)

# TRÁMITES ADMINISTRATIVOS
print("\n--- Reclamo | Administrativo | Simple (Trámite no procesado) ---")
print("Mensaje: Envié los documentos para el cambio de titular hace 2 semanas y no lo procesaron")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Envié los documentos para el cambio de titular hace 2 semanas y no lo procesaron'}, max_tokens=150))
print("-" * 80)

print("\n--- Solicitud | Administrativo | Simple (Cambio titular) ---")
print("Mensaje: Necesito cambiar la titularidad del servicio a mi nombre")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Necesito cambiar la titularidad del servicio a mi nombre'}, max_tokens=150))
print("-" * 80)

print("\n--- Consulta | Administrativo | Simple (Requisitos) ---")
print("Mensaje: ¿Qué documentos necesito para cambiar el domicilio de facturación?")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': '¿Qué documentos necesito para cambiar el domicilio de facturación?'}, max_tokens=150))
print("-" * 80)

print("\n--- Solicitud | Administrativo | Simple (Descuento) ---")
print("Mensaje: Necesito un descuento porque soy jubilado")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Necesito un descuento porque soy jubilado'}, max_tokens=150))
print("-" * 80)

# ============================================================================
# INSTALACIÓN (4 casos)
# ============================================================================
print("\n" + "="*80)
print(">>> TIPO: INSTALACIÓN")
print("="*80)

print("\n--- Reclamo | Instalación | Simple ---")
print("Mensaje: El técnico de instalación no vino en el horario acordado")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'El técnico de instalación no vino en el horario acordado'}, max_tokens=150))
print("-" * 80)

print("\n--- Reclamo | Instalación | Complejo ---")
print("Mensaje: Hace 15 días pedí instalación, vinieron 2 veces y no terminaron. Trabajo remoto y no puedo seguir esperando")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Hace 15 días pedí instalación, vinieron 2 veces y no terminaron. Trabajo remoto y no puedo seguir esperando'}, max_tokens=150))
print("-" * 80)

print("\n--- Solicitud | Instalación | Simple ---")
print("Mensaje: Me mudé y necesito que instalen el servicio en mi nueva dirección")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Me mudé y necesito que instalen el servicio en mi nueva dirección'}, max_tokens=150))
print("-" * 80)

print("\n--- Consulta | Instalación | Simple ---")
print("Mensaje: ¿Cuánto demora la instalación en un barrio nuevo?")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': '¿Cuánto demora la instalación en un barrio nuevo?'}, max_tokens=150))
print("-" * 80)

# ============================================================================
# BAJAS (4 casos)
# ============================================================================
print("\n" + "="*80)
print(">>> TIPO: BAJAS")
print("="*80)

print("\n--- Reclamo | Bajas | Simple ---")
print("Mensaje: Pedí la baja hace una semana y todavía no recibí la confirmación")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Pedí la baja hace una semana y todavía no recibí la confirmación'}, max_tokens=150))
print("-" * 80)

print("\n--- Reclamo | Bajas | Complejo ---")
print("Mensaje: Pedí la baja hace un mes y me siguen cobrando. Si no lo solucionan HOY inicio acciones legales")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Pedí la baja hace un mes y me siguen cobrando. Si no lo solucionan HOY inicio acciones legales'}, max_tokens=150))
print("-" * 80)

print("\n--- Solicitud | Bajas | Simple ---")
print("Mensaje: Quiero dar de baja el servicio porque me mudo al exterior")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'Quiero dar de baja el servicio porque me mudo al exterior'}, max_tokens=150))
print("-" * 80)

print("\n--- Consulta | Bajas | Simple ---")
print("Mensaje: ¿Hay penalidad si doy de baja antes del año?")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': '¿Hay penalidad si doy de baja antes del año?'}, max_tokens=150))
print("-" * 80)

# ============================================================================
# RESUMEN
# ============================================================================
print("\n" + "="*80)
print("FIN DE PRUEBAS V4 FINAL")
print("="*80)
print("\n📊 TOTAL DE CASOS PROBADOS: 24")
print("   - 4 tipos: Técnico, Administrativo, Instalación, Bajas")
print("   - Administrativo incluye: Cobranzas + Facturación + Trámites")
print("="*80 + "\n")


PRUEBAS EXHAUSTIVAS V4 - 4 TIPOS (Cobranzas y Facturación → Administrativo)

>>> TIPO: TÉCNICO

--- Reclamo | Técnico | Simple ---
Mensaje: Hace 2 días que no tengo internet
Ejecutando: clasificacion-completa-v4 (v1, score 5)
--------------------------------------------------
Clasificacion: Reclamo - Técnico - Simple
Razón: Problema técnico estándar sin agravantes adicionales.
--------------------------------------------------------------------------------

--- Reclamo | Técnico | Complejo ---
Mensaje: Es la QUINTA VEZ que se corta internet. Tengo reuniones importantes y NADIE soluciona nada
Ejecutando: clasificacion-completa-v4 (v1, score 5)
--------------------------------------------------
Clasificacion: Reclamo - Técnico - Complejo
Razón: Problema reiterado (5ta vez), impacto laboral, frustración extrema
--------------------------------------------------------------------------------

--- Solicitud | Técnico | Simple ---
Mensaje: Podrían enviar un técnico para revisar la señal WiF

In [33]:
# ============================================================================
# PRUEBAS 
# ============================================================================

print("\n" + "="*80)
print("PRUEBAS TEST")
print("="*80)


# test
print("Mensaje: me tienen que instalar mañana ya tengo turnmo pero no vos a estar, podemos cambiar el turno")
print(ejecutar_desde_repo(df_mio, 'clasificacion-completa-v4', {'mensaje': 'me tienen que instalar mañana ya tengo turnmo pero no vos a estar, podemos cambiar el turno'}, max_tokens=150))
print("-" * 80)



PRUEBAS TEST
Mensaje: me tienen que instalar mañana ya tengo turnmo pero no vos a estar, podemos cambiar el turno
Ejecutando: clasificacion-completa-v4 (v1, score 5)
--------------------------------------------------
Clasificacion: Solicitud - Instalación - Simple
Razón:  Se solicita cambio de turno para instalación, sin mencionar problemas previos ni urgencia extrema.
--------------------------------------------------------------------------------
